[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ashkanhszd/m8_data_science_module/blob/main/soluciones/M8_8.19_21_soluciones.ipynb)

# SOLUCIONES · M8_8.19–M8_8.21 · Programación científica, entornos y fundamentos de Python

**Máster en Hidrogeología y Modelación (UB) · Módulo 8: Herramientas y Data Science**

### Objetivos de la tarde

Al terminar esta tarde serás capaz de:

1. explicar por qué la programación es útil en hidrogeología y describir un cálculo como un flujo *entrada → validación → cálculo → salida → comprobación*;
2. trabajar con soltura en un notebook de Google Colab y leer un mensaje de error;
3. entender qué son un intérprete, un paquete, un entorno virtual y un repositorio Git, y por qué importan para la reproducibilidad;
4. consultar la documentación de Python por tu cuenta;
5. escribir pequeños programas con variables, listas, diccionarios, condiciones, bucles y funciones.

### Plan de la tarde

| Sesión | Contenido | Tiempo aprox. |
|---|---|---|
| **M8_8.19** | Por qué programar · flujos de trabajo · de Graphical Modeler a código · Colab · primeros cálculos · errores | 55 min |
| *Descanso* | | 10 min |
| **M8_8.20** | Intérprete, paquetes y documentación · entornos virtuales y Conda · contenedores · proyecto y rutas · Git y GitHub | 50 min |
| *Descanso* | | 10 min |
| **M8_8.21** | Tipos · listas y diccionarios · condiciones · bucles · funciones · objetos · ejercicio integrador | 55 min |

### Cómo leer este notebook

- 💡 **Idea clave**: concepto que conviene recordar.
- 📖 **Documentación**: cómo encontrar la respuesta en la documentación oficial.
- ✏️ **Ejercicio**: práctica en clase. Escribe tu código donde aparece `...`.
- ✅ **Comprobación**: celda que verifica tu resultado. No hace falta modificarla.
- ⚠️ **Error frecuente**: fallo habitual y cómo evitarlo.

## 0. Preparación

1. **Guarda tu propia copia** en *Archivo → Guardar una copia en Drive*. Si no lo haces, perderás los cambios al cerrar la pestaña.
2. Ejecuta la celda siguiente. Puedes pulsar el botón ▶ o usar **Mayúsculas + Intro**.

In [ ]:
# ==================================================================
# PREPARACIÓN DEL ENTORNO · ejecuta siempre esta celda en primer lugar
# ==================================================================
# · En Google Colab: descarga (clona) el repositorio del curso y se sitúa en él.
# · En un ordenador propio: busca la carpeta del repositorio.
# No hace falta entender cada línea todavía; la estructura se explica en M8_8.20.

import os
import subprocess
import sys
from pathlib import Path

URL_REPOSITORIO = "https://github.com/Ashkanhszd/m8_data_science_module.git"
EN_COLAB = "google.colab" in sys.modules

if EN_COLAB:
    ROOT = Path("/content/m8_data_science_module")
    if not ROOT.exists():
        print("Descargando el repositorio del curso...")
        subprocess.run(["git", "clone", "--depth", "1", URL_REPOSITORIO, str(ROOT)], check=True)
else:
    ROOT = None
    for carpeta in [Path.cwd(), *Path.cwd().parents]:
        if (carpeta / "data" / "input").exists():
            ROOT = carpeta
            break
    if ROOT is None:
        raise FileNotFoundError("No se encuentra data/input: abre el notebook desde la carpeta del repositorio.")

os.chdir(ROOT)
INPUT = ROOT / "data" / "input"          # datos originales (no se modifican)
DATABASE = ROOT / "data" / "database"    # base de datos SQLite
OUTPUT = ROOT / "data" / "output"        # resultados que generamos
for subcarpeta in ["tables", "figures", "logs"]:
    (OUTPUT / subcarpeta).mkdir(parents=True, exist_ok=True)

print("Entorno:", "Google Colab" if EN_COLAB else "ordenador local")
print("Carpeta del repositorio:", ROOT)
print("Python", sys.version.split()[0])

### Uso de asistentes de IA en este módulo

Google Colab incluye un asistente de IA (Gemini) que puede explicar código, sugerir correcciones o escribir programas completos. En este módulo **se permite usarlo**, con estas condiciones:

1. **Primero inténtalo tú.** En los ejercicios de clase, el objetivo es que aprendas a razonar. Si el asistente escribe el código, no aprenderás.
2. **Úsalo para entender, no para copiar.** Es especialmente útil para pedir que te explique un mensaje de error o una función.
3. **Debes poder explicar cada línea** de cualquier código que entregues.
4. **Comprueba lo que te dice.** A veces inventa funciones, parámetros o resultados. La documentación oficial es la referencia.

En la sesión M8_8.43 estudiaremos cómo funcionan estos modelos de lenguaje y cómo usarlos de forma responsable en trabajo científico-técnico.

---
# M8_8.19 · Programación científica y flujos computacionales

## 1. ¿Por qué programar en hidrogeología?

Imagina que tienes 12 pozos con una medida mensual durante 5 años, es decir, 720 medidas. Para cada una quieres calcular la cota piezométrica, detectar valores imposibles y dibujar un gráfico por pozo.

Con una hoja de cálculo es posible, pero cada paso se hace a mano: copiar fórmulas, arrastrar celdas, repetir el gráfico doce veces. Si mañana llegan datos nuevos o descubres un error, hay que repetirlo todo.

Con un programa, el procedimiento se escribe **una vez** y después se ejecuta tantas veces como haga falta. Un programa es:

| Ventaja | Qué significa |
|---|---|
| **Repetible** | Las mismas entradas producen siempre el mismo resultado. |
| **Escalable** | Da igual procesar 1 pozo o 1000. |
| **Trazable** | Cada paso queda escrito: se puede revisar qué se hizo y por qué. |
| **Corregible** | Si hay un error, se corrige una vez y se vuelve a ejecutar todo. |

> 💡 **Idea clave:** un programa que termina sin errores **no es necesariamente correcto** desde el punto de vista científico. El ordenador hace exactamente lo que le pedimos, aunque lo que le pidamos no tenga sentido. La interpretación y el control siguen siendo responsabilidad nuestra.

## 2. Un cálculo como flujo de trabajo

Casi cualquier tarea de análisis sigue la misma estructura:

```text
entrada  →  validación  →  cálculo  →  salida  →  comprobación
```

Vamos a usar un ejemplo que aparecerá durante todo el módulo: la **cota piezométrica**.

```text
      z  (cota del terreno) ──►  ▔▔▔▔▔▔▔▔▔|   |▔▔▔▔▔▔▔▔▔   superficie del terreno
                                         |   |    ▲
                                         |   |    │ d  (profundidad del nivel,
                                         |   |    │     medida desde la superficie)
      h  (cota piezométrica) ──► ~~~~~~~~|~~~|~~~~▼~~~~   nivel del agua en el pozo
                                         |   |
      0 m s.n.m. (nivel de referencia) · · · · · · · · ·
```

$$
h = z - d
$$

donde:

- $h$ es la **cota piezométrica** (m s.n.m.);
- $z$ es la **cota del terreno** en la boca del pozo (m s.n.m.);
- $d$ es la **profundidad del nivel** medida desde la superficie (m).

El flujo quedaría así:

- **Entrada:** $z$ y $d$.
- **Validación:** ¿$d$ es un número? ¿es positiva? ¿es menor que la profundidad del pozo?
- **Cálculo:** $h = z - d$.
- **Salida:** mostrar $h$ con sus unidades.
- **Comprobación:** $h$ debe ser menor que $z$.

## 3. De Graphical Modeler a código

En M8_8.16–18 construiste modelos con **Graphical Modeler** de QGIS. Un modelo gráfico y un programa comparten la misma lógica: ambos reciben entradas, les aplican algoritmos en un orden y producen salidas.

| Graphical Modeler | Python |
|---|---|
| Entrada del modelo (capa, número, campo) | Variable o **parámetro** de una función |
| Algoritmo (caja del modelo) | Operación o **función** |
| Flecha entre algoritmos | Orden de las instrucciones; el resultado de un paso es la entrada del siguiente |
| Salida del modelo | Valor devuelto (`return`), tabla o figura guardada |
| Parámetro editable al ejecutar | Argumento que cambiamos al llamar a la función |
| Ejecutar por lotes | Bucle `for` sobre varios pozos o archivos |

Antes de escribir código conviene describir el procedimiento en **pseudocódigo**: pasos ordenados en lenguaje natural, sin preocuparnos todavía por la sintaxis.

```text
RECIBIR cota del terreno z y profundidad del nivel d
SI d no es un número o es negativa
    AVISAR de que el dato debe revisarse
SI NO
    CALCULAR h = z - d
    MOSTRAR h en metros
FIN
```

## 4. Notebook, script, aplicación y API

| Forma de trabajar | Qué es | Cuándo se usa |
|---|---|---|
| **Notebook** (`.ipynb`) | Documento con celdas de texto, código y resultados | Aprender, explorar datos, documentar un análisis |
| **Script** (`.py`) | Archivo de texto con instrucciones que se ejecutan de principio a fin | Procesos repetitivos y automatizados |
| **Aplicación** | Programa con una interfaz para un usuario, como QGIS o Leapfrog | Tareas interactivas |
| **API** | Conjunto de funciones que un programa ofrece para que otro lo controle | Controlar un modelo o descargar datos desde Python |

Dos conceptos más, solo para reconocerlos:

- **Frontend** es la parte que ve el usuario (la ventana de QGIS, una página web).
- **Backend** es la parte que hace el trabajo y gestiona los datos.

En M8_8.34–36 utilizaréis la API de un código de modelización numérica desde Python.

## 5. Google Colab en diez minutos

Un notebook está formado por **celdas**:

- Las **celdas de texto** (Markdown), como esta, contienen explicaciones, tablas y fórmulas.
- Las **celdas de código** contienen instrucciones de Python. Al ejecutarlas, el resultado aparece debajo.

Algunas ideas importantes:

- Cada celda se ejecuta con **Mayúsculas + Intro**.
- El número entre corchetes, `[3]`, indica el **orden** en que se ejecutaron las celdas.
- Las variables creadas quedan guardadas en la memoria del **entorno de ejecución** (*kernel*) mientras la sesión está abierta.
- Si ejecutas las celdas fuera de orden, puedes obtener resultados difíciles de reproducir.
- *Entorno de ejecución → Reiniciar sesión y ejecutar todo* borra la memoria y ejecuta el notebook de arriba abajo. Es la mejor prueba de que tu notebook funciona.
- Si la sesión está inactiva un tiempo, Colab la desconecta y se pierden las variables. Basta con volver a ejecutar las celdas desde el principio.

> ⚠️ **Error frecuente:** si al ejecutar una celda aparece `NameError`, muchas veces se debe a que no se ha ejecutado antes la celda donde se creaba esa variable.

## 6. Primeros cálculos

Python puede usarse como una calculadora. En una celda, el resultado de la **última línea** se muestra automáticamente.

In [ ]:
# Suma, resta, multiplicación y división
112.4 - 8.7

In [ ]:
# Potencia (**) y paréntesis
area_circulo_m2 = 3.1416 * (0.1 ** 2)   # sección de un pozo de 20 cm de diámetro
area_circulo_m2

Para mostrar varios resultados usamos `print()`. Todo lo que sigue a `#` es un **comentario**: Python lo ignora, pero ayuda a quien lee el código.

In [ ]:
print("Hola, hidrogeología")
print(12 * 60, "medidas mensuales en 12 pozos durante 5 años")

### Variables

Una **variable** es un nombre que guarda un valor. El signo `=` significa **asignar**: guarda el valor de la derecha con el nombre de la izquierda. No significa igualdad matemática.

In [ ]:
# Entrada
cota_terreno_m = 112.4        # cota del terreno del pozo P01 (m s.n.m.)
profundidad_nivel_m = 8.7     # profundidad del nivel medida hoy (m)

# Cálculo
cota_piezometrica_m = cota_terreno_m - profundidad_nivel_m

# Salida
print("Cota piezométrica:", cota_piezometrica_m, "m s.n.m.")

Con una **f-string** (una cadena con `f` delante) podemos insertar valores dentro de un texto y controlar los decimales:

In [ ]:
print(f"La cota piezométrica de P01 es {cota_piezometrica_m:.2f} m s.n.m.")

> 💡 **Buenas prácticas para nombrar variables:**
> - usa nombres descriptivos en minúsculas y separados por guiones bajos, como `profundidad_nivel_m`;
> - incluye la unidad en el nombre cuando sea útil (`_m`, `_mm`, `_uScm`);
> - evita tildes, `ñ` y espacios (funcionan, pero causan problemas con otros programas y teclados);
> - un nombre no puede empezar por un número;
> - Python distingue mayúsculas y minúsculas: `Cota` y `cota` son variables distintas.

### ✏️ Ejercicio 1 · Cota piezométrica de P04

El pozo P04 tiene una cota del terreno de **176.2 m s.n.m.**, y hoy se ha medido el nivel a **21.35 m** de profundidad.

1. Crea las variables `cota_terreno_p04` y `profundidad_p04`.
2. Calcula `cota_piezometrica_p04`.
3. Muestra el resultado con dos decimales y sus unidades.

In [ ]:
cota_terreno_p04 = 176.2
profundidad_p04 = 21.35
cota_piezometrica_p04 = cota_terreno_p04 - profundidad_p04

print(f"Cota piezométrica de P04: {cota_piezometrica_p04:.2f} m s.n.m.")

In [ ]:
# ✅ Comprobación
assert abs(cota_piezometrica_p04 - 154.85) < 0.001, "Revisa el cálculo: h = z - d"
print("✔ Correcto")

## 7. Leer un mensaje de error

Los errores son parte normal de la programación. El mensaje de error (*traceback*) indica **qué** ha fallado y **dónde**. Se lee **de abajo arriba**:

```text
---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
<ipython-input-12-3a4f>  in <cell line: 1>()
----> 1 print(profundida_nivel_m)

NameError: name 'profundida_nivel_m' is not defined
```

1. La **última línea** da el tipo de error (`NameError`) y una explicación (el nombre `profundida_nivel_m` no existe).
2. La flecha `---->` señala la línea que ha fallado.
3. En este caso, el nombre está mal escrito: falta una `d`.

| Error | Significado habitual |
|---|---|
| `SyntaxError` | La instrucción no está bien escrita (paréntesis sin cerrar, dos puntos olvidados…) |
| `NameError` | Se usa un nombre que no existe (mal escrito o no ejecutado todavía) |
| `TypeError` | La operación no es posible con ese tipo de dato (por ejemplo, sumar texto y número) |
| `ValueError` | El tipo es correcto, pero el valor no (por ejemplo, convertir `"8,7"` en número) |
| `FileNotFoundError` | No se encuentra el archivo en la ruta indicada |

### ✏️ Provoca y lee tres errores

En cada celda siguiente, **quita el `#`** del principio de la última línea y ejecútala. Lee el mensaje, identifica el tipo de error y después **corrígelo**.

In [ ]:
# 1) NameError: ¿qué nombre está mal escrito?
# print(profundida_nivel_m)

In [ ]:
# 2) TypeError: no se puede sumar texto y número
profundidad_texto = "8.7"
# profundidad_texto + 1

In [ ]:
# 3) SyntaxError: falta algo al final de la línea
# print("Cota:", cota_piezometrica_m

> 📖 **Documentación:** si no entiendes un error, copia su **última línea** en un buscador (por ejemplo, `TypeError: can only concatenate str (not "int") to str`). Casi siempre encontrarás la explicación en la documentación oficial o en foros como Stack Overflow. También puedes pedir al asistente de IA que te **explique** el mensaje.

---
# M8_8.20 · Entornos de desarrollo y reproducibilidad

## 8. Intérprete, terminal, IDE y notebook

| Elemento | Qué es | Ejemplo |
|---|---|---|
| **Intérprete** | Programa que lee y ejecuta instrucciones de Python | `python` |
| **Terminal** | Ventana donde se escriben órdenes al sistema operativo | Terminal (macOS), Anaconda Prompt (Windows) |
| **IDE** | Editor para escribir, ejecutar y depurar código | VS Code, Spyder, PyCharm |
| **Notebook** | Documento interactivo conectado a un intérprete (*kernel*) | Jupyter, Google Colab |

En Colab, el navegador solo muestra el notebook. El código se ejecuta en un **ordenador de Google**:

```text
   tu navegador                         servidor de Google
 ┌───────────────┐   instrucciones   ┌──────────────────────────────┐
 │   notebook    │ ────────────────► │ kernel → intérprete Python   │
 │ (texto+código)│ ◄──────────────── │ + paquetes + archivos        │
 └───────────────┘    resultados     └──────────────────────────────┘
```

In [ ]:
import platform
import sys

print("Versión de Python:", sys.version.split()[0])
print("Ubicación del intérprete:", sys.executable)
print("Sistema operativo:", platform.system())

## 9. Paquetes, importaciones y versiones

Python trae funciones básicas (`print`, `round`, `len`…). Para trabajar con datos se usan **paquetes** escritos por otras personas:

| Paquete | Para qué sirve | Lo veremos en |
|---|---|---|
| **NumPy** | Cálculo numérico con *arrays* | M8_8.22 |
| **pandas** | Tablas de datos (*DataFrames*) | M8_8.23 |
| **Matplotlib** | Gráficos | M8_8.24 |
| **sqlite3** | Bases de datos SQLite (incluido en Python) | M8_8.25 |

Un paquete se **importa** con `import`. El alias (`np`, `pd`, `plt`) es una convención que todo el mundo usa.

Importar **no instala** nada: solo carga en la sesión un paquete que ya está instalado. En Colab, los paquetes de este módulo ya vienen instalados.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib

print("NumPy", np.__version__)
print("pandas", pd.__version__)
print("Matplotlib", matplotlib.__version__)

> 💡 **Idea clave:** las versiones importan. Una función puede cambiar de comportamiento o desaparecer entre versiones de un paquete. Por eso un análisis reproducible debe indicar qué versiones se usaron.

## 10. Cómo consultar la documentación

Ningún programador se sabe todas las funciones de memoria. Lo importante es **saber buscar**. Hay cuatro formas, de más rápida a más completa:

1. **`help(funcion)`** muestra la documentación dentro del notebook.
2. **`funcion?`** (solo en notebooks) abre la ayuda en un panel lateral.
3. **Autocompletado y ayuda contextual:** escribe `np.` y pulsa **Tab**, o sitúa el cursor dentro de los paréntesis de una función para ver sus parámetros.
4. **Documentación oficial en la web**, que incluye ejemplos:
   - Python: https://docs.python.org/es/3/
   - NumPy: https://numpy.org/doc/stable/
   - pandas: https://pandas.pydata.org/docs/
   - Matplotlib: https://matplotlib.org/stable/

La página de documentación de una función suele tener siempre las mismas partes:

- **firma**: nombre y parámetros, por ejemplo `round(number, ndigits=None)`;
- **descripción** de lo que hace;
- **parámetros**: qué significa cada uno y su valor por defecto;
- **valor devuelto**;
- **ejemplos**.

In [ ]:
help(round)

### ✏️ Ejercicio 2 · Aprender una función leyendo su documentación

Usa la ayuda de `round` (la celda anterior o `round?`) para:

1. redondear la cota `112.456` a **un decimal** y guardar el resultado en `cota_redondeada`;
2. averiguar qué devuelve `round(2.5)` y guardar el resultado en `resultado_2_5`. ¿Es lo que esperabas? La documentación en la web explica por qué: https://docs.python.org/es/3/library/functions.html#round

In [ ]:
cota_redondeada = round(112.456, 1)
resultado_2_5 = round(2.5)      # devuelve 2: Python redondea al par más cercano en los empates
print(cota_redondeada, resultado_2_5)

In [ ]:
# ✅ Comprobación
assert cota_redondeada == 112.5, "Revisa el segundo parámetro de round (ndigits)"
assert resultado_2_5 == 2, "Ejecuta round(2.5) y observa el resultado"
print("✔ Correcto. Python redondea los empates al número par más cercano ('redondeo del banquero').")

## 11. Entornos virtuales y Conda

**El problema.** Un proyecto de 2022 funciona con pandas 1.5, y uno nuevo necesita pandas 3.0. Si ambos usan la misma instalación de Python, actualizar pandas para el proyecto nuevo puede romper el antiguo.

**La solución** es un **entorno virtual**: una instalación aislada de Python con sus propios paquetes y versiones. Cada proyecto tiene su entorno.

```text
 ordenador
 ├── entorno "proyecto_2022"  →  Python 3.10 + pandas 1.5
 ├── entorno "modulo8"        →  Python 3.12 + numpy + pandas + matplotlib
 └── entorno "tfm"            →  Python 3.12 + flopy + ...
```

**Conda** (incluido en Anaconda) crea entornos e instala paquetes. **pip** es el instalador de paquetes estándar de Python.

Los comandos siguientes **se escriben en una terminal** (Anaconda Prompt en Windows, Terminal en macOS o Linux), no en una celda de Python. En clase los ejecutaremos en directo en la terminal:

```bash
# ¿Está instalado conda? ¿Qué versión?
conda --version

# Ver los entornos que existen (el activo aparece con *)
conda env list

# Crear un entorno llamado modulo8 con Python 3.12
conda create --name modulo8 python=3.12

# Activarlo: a partir de aquí, python y los paquetes son los de este entorno
conda activate modulo8

# Instalar paquetes en el entorno activo
conda install -c conda-forge numpy pandas matplotlib openpyxl jupyter

# Ver qué paquetes y versiones contiene
conda list

# Guardar la "receta" del entorno en un archivo
conda env export --from-history > environment.yml

# Reconstruir el mismo entorno en otro ordenador a partir de la receta
conda env create --file environment.yml

# Salir del entorno
conda deactivate
```

El archivo `environment.yml` es la **receta** del entorno. El repositorio del curso incluye uno:

In [ ]:
print((ROOT / "environment.yml").read_text())

> 💡 **En Colab** no hay Conda. Cada sesión es un entorno temporal con muchos paquetes ya instalados. Si falta alguno, se instala solo para esa sesión con `%pip install nombre_paquete` en una celda. En tu ordenador, lo recomendable es usar un entorno Conda por proyecto, como explica la *Guía de instalación de Anaconda* del curso.

## 12. Contenedores y Docker

Un entorno virtual aísla Python y sus paquetes. Un **contenedor** va un paso más allá: empaqueta también bibliotecas del sistema operativo y otros programas. Así, el mismo contenedor funciona igual en cualquier ordenador o servidor.

```text
 entorno virtual:   proyecto → Python + paquetes
 contenedor:        proyecto → Python + paquetes + bibliotecas del sistema + otros programas
```

**Docker** es la herramienta más utilizada para crear y ejecutar contenedores. Se usa, por ejemplo, para distribuir un código de modelización con todas sus dependencias o para ejecutar un servicio web. En este módulo solo necesitas reconocer el concepto; no crearemos contenedores.

## 13. Estructura de un proyecto y rutas

Un proyecto ordenado separa **datos**, **código** y **resultados**. Esta es la estructura del repositorio del curso:

```text
m8_data_science_module/
├── README.md                  instrucciones generales
├── environment.yml            receta del entorno Conda
├── data/
│   ├── input/                 datos originales (NO se modifican)
│   ├── database/              base de datos SQLite
│   └── output/                resultados que se pueden regenerar
│       ├── tables/
│       ├── figures/
│       └── logs/
├── M8_8.19_21_.../            notebooks de cada tarde
├── ...
└── soluciones/                soluciones de los ejercicios de clase
```

Principios:

1. **Los datos originales no se sobrescriben nunca.** Las versiones corregidas se guardan en `data/output`.
2. **Las rutas son relativas a la raíz del proyecto**, no a tu ordenador. Una ruta como `C:/Users/Ana/Escritorio/pozos.csv` solo funciona en el ordenador de Ana.
3. **Todo resultado debe poder regenerarse** ejecutando el código.

En Python, las rutas se manejan con `pathlib.Path`. La celda de preparación definió `ROOT` (raíz del repositorio) e `INPUT` (datos de entrada). El operador `/` une partes de una ruta en cualquier sistema operativo:

In [ ]:
print("Raíz:", ROOT)
print("Datos de entrada:", INPUT)
print()
print("Archivos en data/input:")
for ruta in sorted(INPUT.iterdir()):
    print("  ", ruta.name)

### ✏️ Ejercicio 3 · Una ruta portable

1. Construye la ruta al archivo `pozos.csv` a partir de `INPUT` y guárdala en `ruta_pozos`.
2. Comprueba si existe con el método `.exists()`.
3. Muestra el contenido del archivo con `.read_text()`.

In [ ]:
ruta_pozos = INPUT / "pozos.csv"
print("¿Existe?", ruta_pozos.exists())
print(ruta_pozos.read_text())

In [ ]:
# ✅ Comprobación
assert ruta_pozos == INPUT / "pozos.csv", "La ruta debe construirse como INPUT / 'pozos.csv'"
assert ruta_pozos.exists()
print("✔ Correcto")

## 14. Control de versiones: Git y GitHub

Seguro que conoces esto: `informe_final.docx`, `informe_final_v2.docx`, `informe_final_v2_ESTE.docx`… Un **sistema de control de versiones** resuelve este problema: guarda el historial completo de cambios de un proyecto, quién los hizo y por qué.

| Concepto | Qué es |
|---|---|
| **Git** | Programa de control de versiones. Funciona en tu ordenador. |
| **GitHub** | Servicio web que aloja repositorios Git para compartirlos y colaborar |
| **Repositorio** | El proyecto junto con todo su historial |
| **Commit** | "Fotografía" del proyecto en un momento dado, con un mensaje que explica el cambio |
| **Clonar** | Descargar una copia completa de un repositorio |
| **Push / Pull** | Enviar tus commits al repositorio remoto / traer los cambios de otros |

El ciclo de trabajo habitual es:

```text
modificar archivos → revisar cambios → seleccionar (add) → registrar (commit) → publicar (push)
```

En clase veremos estos comandos en la terminal. Se escriben en la terminal, no en Python:

```bash
git clone https://github.com/Ashkanhszd/m8_data_science_module.git   # descargar el repositorio
git status                                   # ¿qué archivos han cambiado?
git diff                                     # ¿qué ha cambiado exactamente?
git add notebook.ipynb                       # seleccionar un cambio
git commit -m "Añade el cálculo de la cota"  # registrarlo con un mensaje
git log --oneline                            # ver el historial
git pull                                     # traer cambios del repositorio remoto
git push                                     # publicar mis cambios
```

**¿Cómo ha llegado este notebook a tu pantalla?** El notebook está en un repositorio público de GitHub. Colab lo abre directamente desde allí, y la celda de preparación **clona** el repositorio para tener también los datos. Los cambios que hagas en Colab **no modifican** el repositorio original: por eso debes guardar una copia en Drive.

> ⚠️ **Nunca subas a GitHub** contraseñas, claves de acceso, datos personales o datos confidenciales de un cliente. Un repositorio público lo puede ver cualquier persona.

## 15. Reproducibilidad: lista mínima

Un análisis es **reproducible** si otra persona (o tú dentro de un año) puede obtener el mismo resultado a partir de los mismos datos. Para ello:

- [ ] los datos originales se conservan sin modificar;
- [ ] las rutas son relativas a la raíz del proyecto;
- [ ] se documentan los paquetes y sus versiones;
- [ ] el código se ejecuta de principio a fin sin pasos manuales;
- [ ] los parámetros y decisiones están escritos, no en la memoria de nadie;
- [ ] los resultados se pueden regenerar;
- [ ] las unidades y supuestos están documentados.

> 💡 Reproducible no significa correcto: un análisis puede ser perfectamente reproducible y estar mal planteado. Pero si es reproducible, se puede **revisar**.

---
# M8_8.21 · Fundamentos de Python

## 16. Tipos de datos básicos

Cada valor en Python tiene un **tipo**, y el tipo determina qué operaciones son posibles.

| Tipo | Qué representa | Ejemplo |
|---|---|---|
| `int` | número entero | `12` (número de pozos) |
| `float` | número decimal | `8.7` (profundidad en m) |
| `str` | texto (*string*), entre comillas | `"P01"`, `"Aluvial"` |
| `bool` | verdadero o falso | `True`, `False` |
| `None` | ausencia de valor | `None` |

La función `type()` indica el tipo de un valor.

In [ ]:
numero_pozos = 12
profundidad_m = 8.7
id_pozo = "P01"
es_valida = True
observacion = None

for valor in [numero_pozos, profundidad_m, id_pozo, es_valida, observacion]:
    print(repr(valor), "→", type(valor).__name__)

### Ausencia de datos: `None`, `NaN`, cero y texto vacío

En datos reales es frecuente que falte un valor. **No es lo mismo** un dato ausente que un cero:

| Valor | Significado |
|---|---|
| `0.0` | se midió y el resultado fue cero (por ejemplo, un mes sin lluvia) |
| `None` | no hay valor (objeto vacío de Python) |
| `NaN` (*Not a Number*) | valor numérico ausente; es la forma estándar de representar ausentes en NumPy y pandas |
| `""` | texto vacío |

`NaN` tiene una propiedad curiosa: **no es igual ni a sí mismo**. Por eso, para detectarlo se usan funciones específicas como `math.isnan()`, o `isna()` en pandas.

In [ ]:
import math

nan = float("nan")
print("0.0 == None →", 0.0 == None)
print("nan == nan  →", nan == nan)
print("math.isnan(nan) →", math.isnan(nan))
print("observacion is None →", observacion is None)

## 17. Conversión de tipos

`int()`, `float()` y `str()` convierten un valor en otro tipo **cuando es posible**. Es habitual cuando los datos llegan como texto, por ejemplo desde un formulario o una hoja de campo.

In [ ]:
profundidad_texto = "8.7"
profundidad_numero = float(profundidad_texto)
print(profundidad_numero + 1)

# En España solemos escribir la coma decimal. float("8,7") produce un ValueError:
profundidad_con_coma = "8,7"
profundidad_corregida = float(profundidad_con_coma.replace(",", "."))
print(profundidad_corregida)

## 18. Operadores

| Tipo | Operadores | Ejemplo |
|---|---|---|
| Aritméticos | `+  -  *  /  **` (potencia)  `//` (división entera)  `%` (resto) | `12 * 86.4` |
| Comparación | `==  !=  <  <=  >  >=` | `profundidad > 20` |
| Lógicos | `and  or  not` | `d >= 0 and d < 80` |

Una comparación siempre produce un valor `bool` (`True` o `False`).

> ⚠️ `=` **asigna**; `==` **compara**.

Ejemplo: un caudal de 12 L/s expresado en m³/día. Un día tiene 86 400 s y 1 m³ son 1000 L, así que:

$$
Q\,[\mathrm{m^3/d}] = Q\,[\mathrm{L/s}] \times \frac{86\,400}{1000} = Q\,[\mathrm{L/s}] \times 86.4
$$

In [ ]:
caudal_l_s = 12
caudal_m3_d = caudal_l_s * 86.4
print(f"{caudal_l_s} L/s = {caudal_m3_d:.1f} m³/día")

profundidad_m = 24.5
profundidad_pozo_m = 80
print("¿Nivel profundo (> 20 m)?", profundidad_m > 20)
print("¿Dentro del pozo?", profundidad_m >= 0 and profundidad_m < profundidad_pozo_m)

## 19. Listas

Una **lista** guarda varios valores en orden. Se escribe entre corchetes y se puede modificar. Las posiciones (**índices**) empiezan en **0**.

```text
niveles  =  [ 7.9,  7.8,  7.7,  7.5,  7.6 ]
índice        0     1     2     3     4
índice       -5    -4    -3    -2    -1
```

In [ ]:
niveles_p01 = [7.9, 7.8, 7.7, 7.5, 7.6]   # profundidad del nivel (m), enero a mayo

print("Primer valor:", niveles_p01[0])
print("Último valor:", niveles_p01[-1])
print("Los tres primeros:", niveles_p01[0:3])   # el índice final (3) no se incluye
print("Número de valores:", len(niveles_p01))
print("Mínimo y máximo:", min(niveles_p01), max(niveles_p01))
print("Media:", sum(niveles_p01) / len(niveles_p01))

niveles_p01.append(7.9)       # añadir la medida de junio
print("Lista actualizada:", niveles_p01)

> 📖 **Documentación:** las listas tienen muchos métodos útiles (`append`, `sort`, `count`…). Están todos en el tutorial oficial: https://docs.python.org/es/3/tutorial/datastructures.html. ¿Cómo ordenarías `niveles_p01` de mayor a menor sin modificar la lista original? (Pista: busca `sorted` y su parámetro `reverse`.)

In [ ]:
sorted(niveles_p01, reverse=True)

## 20. Diccionarios

Un **diccionario** guarda pares **clave → valor**. Es ideal para describir una entidad, como la ficha de un pozo. Se escribe entre llaves `{}`, y cada clave debe ser única.

In [ ]:
pozo = {
    "id_pozo": "P01",
    "acuifero": "Aluvial",
    "cota_terreno_m": 112.4,
    "profundidad_pozo_m": 30.0,
}

print("Acuífero:", pozo["acuifero"])
pozo["uso"] = "control"          # añadir una clave nueva
print(pozo)

# .get() devuelve un valor por defecto si la clave no existe (en lugar de dar error)
print("Responsable:", pozo.get("responsable", "sin asignar"))

Una **tupla** es como una lista que no se puede modificar. Se escribe entre paréntesis y se usa para valores que van juntos, como unas coordenadas: `coordenadas = (421100, 4582100)`.

## 21. Condicionales: `if`, `elif`, `else`

Permiten ejecutar un bloque u otro según una condición. La **indentación** (4 espacios) indica qué líneas pertenecen a cada bloque. En Python, la indentación es parte de la sintaxis: no es opcional.

In [ ]:
profundidad = 24.5

if profundidad < 0:
    clase = "valor imposible: revisar"
elif profundidad < 10:
    clase = "nivel somero"
elif profundidad < 20:
    clase = "nivel intermedio"
else:
    clase = "nivel profundo"

print(f"{profundidad} m → {clase}")

## 22. Bucles `for`

Un bucle repite un bloque de instrucciones para cada elemento de una colección. Es el equivalente al "procesamiento por lotes" de Graphical Modeler.

In [ ]:
cotas_terreno = {"P01": 112.4, "P02": 108.7, "P03": 105.9}
profundidades = {"P01": 7.9, "P02": 8.2, "P03": 9.6}

for id_pozo in cotas_terreno:
    h = cotas_terreno[id_pozo] - profundidades[id_pozo]
    print(f"{id_pozo}: h = {h:.2f} m s.n.m.")

Un patrón muy frecuente consiste en **acumular** un resultado dentro de un bucle: contar, sumar o guardar en una lista.

In [ ]:
niveles_p03 = [8.4, 8.9, 9.4, 9.9, 10.2, 9.7, 9.3, 8.8]
umbral_m = 9.5

numero_superan = 0
for nivel in niveles_p03:
    if nivel > umbral_m:
        numero_superan = numero_superan + 1

print(f"{numero_superan} de {len(niveles_p03)} medidas superan {umbral_m} m")

### ✏️ Ejercicio 4 · Recorrer una serie

Con la lista `niveles_p03` de la celda anterior:

1. calcula la **suma** de todas las medidas **con un bucle** (sin usar `sum`) y guárdala en `suma`;
2. calcula la media y guárdala en `media`;
3. crea una lista `diferencias` con la diferencia de cada medida respecto a la media.

In [ ]:
suma = 0
for nivel in niveles_p03:
    suma = suma + nivel

media = suma / len(niveles_p03)

diferencias = []
for nivel in niveles_p03:
    diferencias.append(nivel - media)

print(suma, media, diferencias)

In [ ]:
# ✅ Comprobación
assert abs(suma - sum(niveles_p03)) < 1e-9, "Revisa la acumulación de la suma"
assert abs(media - 9.325) < 1e-9, "La media es la suma dividida por el número de valores"
assert len(diferencias) == len(niveles_p03) and abs(sum(diferencias)) < 1e-9
print("✔ Correcto. Observa que la suma de las diferencias respecto a la media es cero.")

## 23. Funciones

Una **función** agrupa unas instrucciones con un nombre para reutilizarlas. Es el equivalente a construir un modelo en Graphical Modeler y ejecutarlo tantas veces como haga falta.

```python
def nombre_funcion(parametro_1, parametro_2):
    """Docstring: qué hace la función, qué recibe y qué devuelve."""
    resultado = ...
    return resultado
```

- `def` inicia la definición.
- Los **parámetros** son los nombres de las entradas.
- Los **argumentos** son los valores concretos que se pasan al **llamar** a la función.
- `return` devuelve el resultado a quien llamó a la función.
- La **docstring** (texto entre triples comillas) documenta la función. `help()` la muestra.

In [ ]:
def calcular_cota_piezometrica(cota_terreno_m, profundidad_nivel_m):
    """Calcula la cota piezométrica h = z - d (en metros).

    Si la profundidad es negativa, lanza un ValueError: es un valor imposible
    que debe revisarse antes de continuar.
    """
    if profundidad_nivel_m < 0:
        raise ValueError(f"Profundidad negativa ({profundidad_nivel_m} m): revisa el dato.")
    return cota_terreno_m - profundidad_nivel_m


h_p01 = calcular_cota_piezometrica(112.4, 8.7)
h_p04 = calcular_cota_piezometrica(cota_terreno_m=176.2, profundidad_nivel_m=21.35)
print(h_p01, h_p04)

In [ ]:
# Nuestra propia función ya tiene documentación:
help(calcular_cota_piezometrica)

> ⚠️ **`print` no es lo mismo que `return`.** `print` muestra un valor en pantalla, pero no lo entrega. Si una función solo usa `print`, el resultado no se puede guardar ni reutilizar: la función devuelve `None`.

In [ ]:
def media_que_imprime(valores):
    print(sum(valores) / len(valores))

def media_que_devuelve(valores):
    return sum(valores) / len(valores)

a = media_que_imprime([1, 2, 3])
b = media_que_devuelve([1, 2, 3])
print("a =", a, "| b =", b)

## 24. Objetos, atributos y métodos

En Python, **todo es un objeto**: un número, un texto, una lista y, más adelante, una tabla de pandas. Un objeto tiene:

- **atributos**: características que se consultan sin paréntesis, como `tabla.shape` (dimensiones de una tabla);
- **métodos**: acciones que se ejecutan con paréntesis, como `texto.upper()` o `lista.append(7.9)`.

El tipo de un objeto (su **clase**) define qué atributos y métodos tiene. Por eso la documentación de pandas se organiza por clases (`DataFrame`, `Series`…). En este módulo no escribiremos clases propias, pero sí usaremos continuamente objetos y sus métodos.

In [ ]:
id_leido = "  p01 "
print(repr(id_leido.strip().upper()))       # métodos de str: quitar espacios y pasar a mayúsculas

numero = 7.5
print(numero.is_integer())                  # método de float

# Autocompletado: escribe id_leido. y pulsa Tab para ver todos los métodos de un texto

## 25. Scripts y módulos (avance)

El código de un notebook también puede guardarse en un archivo `.py`:

- si se ejecuta de principio a fin, es un **script**;
- si contiene funciones que se importan desde otro programa (`from mis_funciones import calcular_cota_piezometrica`), es un **módulo**.

Así se reutilizan las mismas funciones en muchos notebooks sin copiar y pegar. Lo haremos en la práctica en M8_8.31.

## ✏️ Ejercicio integrador · Hoja de campo

Un técnico ha pasado a limpio las medidas de una campaña. Los datos llegan como una **lista de diccionarios**, con los problemas típicos de un dato real:

- algunas profundidades están escritas como texto con **coma decimal**;
- falta un valor (`None`);
- hay una profundidad **negativa**, que es imposible.

Completa la función `procesar_registro` para que, para cada registro:

1. si la profundidad es `None`, devuelva el estado `"sin dato"`;
2. convierta la profundidad a `float` (sustituyendo la coma por un punto si es texto);
3. si la profundidad es negativa, devuelva el estado `"revisar"`;
4. en caso contrario, calcule la cota piezométrica con `calcular_cota_piezometrica` y devuelva el estado `"valido"`.

La función devuelve un diccionario con `id_pozo`, `estado` y `cota_piezometrica_m` (`None` si no se ha podido calcular).

In [ ]:
cotas_terreno = {"P01": 112.4, "P02": 108.7, "P03": 105.9, "P04": 176.2, "P05": 190.5}

hoja_campo = [
    {"id_pozo": "P01", "profundidad": 7.85},
    {"id_pozo": "P02", "profundidad": "8,12"},
    {"id_pozo": "P03", "profundidad": None},
    {"id_pozo": "P04", "profundidad": "21,40"},
    {"id_pozo": "P05", "profundidad": -2.3},
]

In [ ]:
def procesar_registro(registro, cotas_terreno):
    """Valida una medida de campo y calcula su cota piezométrica."""
    id_pozo = registro["id_pozo"]
    profundidad = registro["profundidad"]

    # 1. ¿Falta el dato?
    if profundidad is None:
        return {"id_pozo": id_pozo, "estado": "sin dato", "cota_piezometrica_m": None}

    # 2. Convertir a número (el texto puede tener coma decimal)
    if isinstance(profundidad, str):
        profundidad = float(profundidad.replace(",", "."))

    # 3. ¿Es un valor imposible?
    if profundidad < 0:
        return {"id_pozo": id_pozo, "estado": "revisar", "cota_piezometrica_m": None}

    # 4. Cálculo
    h = calcular_cota_piezometrica(cotas_terreno[id_pozo], profundidad)
    return {"id_pozo": id_pozo, "estado": "valido", "cota_piezometrica_m": round(h, 2)}


resultados = []
for registro in hoja_campo:
    resultados.append(procesar_registro(registro, cotas_terreno))

for r in resultados:
    print(r)

In [ ]:
# ✅ Comprobación
estados = [r["estado"] for r in resultados]
assert estados == ["valido", "valido", "sin dato", "valido", "revisar"], f"Estados obtenidos: {estados}"
assert abs(resultados[1]["cota_piezometrica_m"] - 100.58) < 0.01, "Revisa la conversión de '8,12'"
print("✔ Correcto")

## Síntesis

- Un programa científico sigue el flujo **entrada → validación → cálculo → salida → comprobación**.
- Un modelo de Graphical Modeler y una función de Python comparten la misma lógica.
- En un notebook importa el **orden de ejecución**. *Reiniciar y ejecutar todo* es la prueba de que el notebook funciona.
- Los errores se leen **de abajo arriba**, y la documentación se consulta con `help()`, `?`, Tab y la web oficial.
- Los **entornos virtuales** fijan paquetes y versiones. **Git** registra la historia del proyecto y **GitHub** la comparte.
- Tipos, listas, diccionarios, condiciones, bucles y funciones son las piezas básicas de cualquier programa.

### Lo que viene en M8_8.22–24

La lista de diccionarios del ejercicio integrador es, en realidad, una **tabla**. En la próxima sesión la convertiremos en un *DataFrame* de pandas y trabajaremos con los 720 registros reales del curso.

In [ ]:
pd.DataFrame(resultados)